# Model Diagnostic Validation & Hard-Negative Mining

This notebook performs quantitative diagnostic evaluation of the trained **`DenseDetector`** model on synthetic test spectra.

### Workflow:
1. **Model & Config Ingestion**: Loads the serialized model weights and geometry configuration.
2. **Batch Diagnostic Evaluation**: Runs peak-level matching (greedy 1D nearest-neighbor) to compute Precision, Recall, and F1-score.
3. **Failure Analysis**: Categorizes failure modes into False Negatives (missed peaks / grid collisions) and False Positives (spurious noise activations).
4. **Hard-Mined Dataset Export**: Aggregates all difficult/failed spectra into `hard_mined_v1.npz` for targeted retraining (`main_rebuild.py`).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os

# Add project root to sys.path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import torch

from src.signal_sample_module import generate_dataset
from src.validation import evaluate_model, build_hard_mined_set
from src.inference import load_model

# Select execution device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Load Model and Configurations

In [ ]:
# Model and config paths
model_path = '../saved_models/dense_model.pt'
config_path = '../saved_models/dense_model_config.json'

# Load model weights and config dataclass
cnn_model, cfg, device = load_model(model_path, config_path, device=device, mode="eval")
print(f"Loaded DenseDetector: grid points = {cfg.n_points}, downsampling stride = {cfg.stride}, grid cells = {cfg.l_grid}")

## 2. Generate Synthetic Test Dataset & Evaluate

In [ ]:
# Generate synthetic test dataset with Poisson noise
n_test = 8000
rng = np.random.default_rng(0)

print(f"Generating {n_test} synthetic test spectra...")
X_test, peaks_test = generate_dataset(n_test, rng, cfg)
X_test = torch.tensor(X_test, dtype=torch.float32)

# Extract raw intensity channel (shape: [N, n_points])
X_test_raw = X_test[:, 0, :] if X_test.dim() > 2 else X_test

# Run peak-level diagnostic matching (default tolerance: half a grid cell)
print("Evaluating model detections against ground truth...")
results = evaluate_model(cnn_model, cfg, device, X_test_raw, peaks_test)

summary = results["summary"]
print("\n=== Validation Metrics Summary ===")
print(f"Test Samples: {summary['n_samples']}")
print(f"True Positives (TP):  {summary['tp']}")
print(f"False Negatives (FN): {summary['fn']}")
print(f"False Positives (FP): {summary['fp']}")
print(f"Precision: {summary['precision']:.4f}")
print(f"Recall:    {summary['recall']:.4f}")
print(f"F1-Score:  {summary['f1']:.4f}")

## 3. Diagnostic Breakdown of Failure Modes

In [ ]:
fn_records = results["fn_records"]
fp_records = results["fp_records"]

print(f"Total Missed Peak Records (FN): {len(fn_records)}")
print(f"Total Spurious Prediction Records (FP): {len(fp_records)}")

# Analyze False Negatives: check how many are due to grid cell collisions
if fn_records:
    grid_collisions = sum(1 for r in fn_records if r.get("grid_collision", False))
    print(f"  - FN caused by grid collision (>=2 true peaks in same cell): {grid_collisions} ({100 * grid_collisions / len(fn_records):.1f}%)")
    print(f"  - FN due to sub-threshold confidence: {len(fn_records) - grid_collisions}")

# Analyze False Positives: average confidence of false alarms
if fp_records:
    avg_fp_conf = np.mean([r["confidence"] for r in fp_records])
    print(f"  - Average confidence of FP detections: {avg_fp_conf:.3f}")

## 4. Build and Export Hard-Mined Dataset for Retraining

In [ ]:
# Collect unique profiles that produced errors (FN or FP)
X_hard, peaks_hard = build_hard_mined_set(fn_records, fp_records, include_fp_profiles=True)

export_path = "../hard_mined_v1.npz"
np.savez(export_path, X=X_hard, peaks=np.array(peaks_hard, dtype=object))

print(f"Successfully exported {len(X_hard)} hard-mined spectra to: {export_path}")
print("Use `main_rebuild.py` to retrain/fine-tune the model with oversampled hard cases.")